In [1]:
# %pip install ninja --break-system-packages

In [2]:
# %pip install --upgrade transformers --break-system-packages

In [3]:
# %pip install git+https://github.com/intel/auto-round.git --break-system-packages

In [4]:
# %pip install git+https://github.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [5]:
# %pip install compressed-tensors --break-system-packages

In [6]:
import os

import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from transformers import AutoModelForImageTextToText, AutoProcessor


In [7]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [8]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.14.0+cu132
CUDA Available: True
CUDA Version: 13.2
GPU Name: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GB


In [9]:
notebook_login()

In [10]:
MODEL_ID = "lightonai/LightOnOCR-3-0.8B"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [11]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 15 files:   0%|                                | 0/15 [00:00<?, ?it/s]Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)

Reconstructing (incomplete total...):   0%|       |  0.00B / 9.27kB            
Reconstructing (incomplete total...):   1%|       | 9.27kB / 1.66MB            

Fetching 15 files:   7%|█▌                      | 1/15 [00:00<00:03,  4.34it/s]
Reconstructing (incomplete t

In [12]:
import os

from safetensors import safe_open

for file in os.listdir(LOCAL_PATH):
    if file.endswith(".safetensors"):
        path = os.path.join(LOCAL_PATH, file)
        print(f"\nChecking {file}")

        with safe_open(path, framework="pt") as f:
            keys = list(f.keys())

            mtp_keys = [k for k in keys if "mtp" in k.lower()]
            for k in mtp_keys:
                print(k)


Checking model.safetensors


In [13]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [14]:
model

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 768)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-11): 12 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=768, out_features=3072, bias=True)
            (linear_fc2): Linear(in_features=3072, out_features=768, bias=True)
            (act_fn): GELUTanh()
          )
      

In [15]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [16]:
TUNING_CONFIG = {
    "group_size": 32,
    "sym": True,
    "iters": 1200,  # High accuracy (Production grade)
    "nsamples": 512,  # More calibration data
    "batch_size": 4,
    "seqlen": 4096,
    "low_gpu_mem_usage": False,  # Keep on GPU for speed
    "enable_torch_compile": True,  # JIT acceleration
    "quant_nontext_module": False,  # Keep Vision Tower in FP16 (Crucial for VLM accuracy)
}

In [17]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W4A16",
    **TUNING_CONFIG,
)

2026-10-10 19:24:04 WARNING autoround.py L494: Passing 'group_size' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-10-10 19:24:04 WARNING autoround.py L494: Passing 'sym' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-10-10 19:24:04 WARNING autoround.py L494: Passing 'iters' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.


In [18]:
# SINGLE CALL to save all 3 formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round,auto_gptq,llm_compressor", inplace=True
)

2026-10-10 19:24:05 WARNING logging.py L340: some layers are skipped quantization (shape not divisible by 32): model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22].linear_attn.in_proj_a, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22].linear_attn.in_proj_b
2026-10-10 19:24:05 INFO base.py L1362: `torch.compile` is enabled
2026-10-10 19:24:06 INFO orchestrator.py L594: start to cache block inputs
2026-10-10 19:24:06 INFO mllm.py L88: Using MLLM template: qwen3_5
2026-10-10 19:24:06 INFO mllm.py L127: Multimodal model with non-MLLM calibration dataset 'NeelNanda/pile-10k' and quant_nontext_module=False: using the standard text dataloader (vision/audio towers are not being quantized, so text-only calibration through the full-model forward is sufficient).
2026-10-10 19:24:06 INFO calib_dataset.py L1227: Preprocessing calibration dataset in a subprocess to avoid memory leaks...
2026-10-10 19:24:09 INFO device.py L1553: 'peak_ram': 7.63GB, 'peak_vram': 1.65GB
2026-10-10 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

packing: 100%|██████████| 186/186 [00:00<00:00, 231.18it/s]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-10-10 19:34:00 INFO device.py L1553: 'peak_ram': 20.14GB, 'peak_vram': 18.4GB


(Qwen3_5ForConditionalGeneration(
   (model): Qwen3_5Model(
     (visual): Qwen3_5VisionModel(
       (patch_embed): Qwen3_5VisionPatchEmbed(
         (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
       )
       (pos_embed): Embedding(2304, 768)
       (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
       (blocks): ModuleList(
         (0-11): 12 x Qwen3_5VisionBlock(
           (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
           (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
           (attn): Qwen3_5VisionAttention(
             (qkv): Linear(in_features=768, out_features=2304, bias=True)
             (proj): Linear(in_features=768, out_features=768, bias=True)
           )
           (mlp): Qwen3_5VisionMLP(
             (linear_fc1): Linear(in_features=768, out_features=3072, bias=True)
             (linear_fc2): Linear(in_features=3072, out_features=768, bias=True)
             (act_fn): GELUTanh(

In [19]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [20]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/auto-round-auto-gptq"),
        f"{base_name}-W4A16-AutoRound",
        hf_token,
    )
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/auto-gptq"),
        f"{base_name}-W4A16-AutoRound-GPTQ",
        hf_token,
    )
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/llm-compressor-wint-a16"),
        f"{base_name}-W4A16-AutoRound-LLM-Compressor",
        hf_token,
    )

else:
    print("No Hugging Face token found. Skipping upload to hub.")


[Hub] Pushing ./AutoRound/local_model-w4g32/auto-round-auto-gptq to Vishva007/LightOnOCR-3-0.8B-W4A16-AutoRound...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/LightOnOCR-3-0.8B-W4A16-AutoRound

[Hub] Pushing ./AutoRound/local_model-w4g32/auto-gptq to Vishva007/LightOnOCR-3-0.8B-W4A16-AutoRound-GPTQ...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/LightOnOCR-3-0.8B-W4A16-AutoRound-GPTQ

[Hub] Pushing ./AutoRound/local_model-w4g32/llm-compressor-wint-a16 to Vishva007/LightOnOCR-3-0.8B-W4A16-AutoRound-LLM-Compressor...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/LightOnOCR-3-0.8B-W4A16-AutoRound-LLM-Compressor
